---
title: "DRG Cleaning v2"

author: "Carlos Resurreccion"

date: "2024-10-21"

---


# Parameters

Change which year to process in
`~/drg-pipeline/data-cleaning/cache/year_to_load.txt`

Change other rarely touched parameters in
`~/drg-pipeline/data-cleaning/r_scripts_v2/00_v2_params-fpaths.R`


In [10]:
# Delete all R objects and run garbage collection so we start with a clean slate
rm(list = ls())
invisible(gc())

# Whether to sample each split_part by sample_size_divisor
# (useful when iterating through code runs in quick succession)
to_sample <- FALSE
# TODO: Add description here
to_write <- TRUE
# TODO: Add description here
to_flush <- FALSE
# TODO: Add description here
to_parallel <- as.logical(Sys.getenv("TO_PARALLEL", "TRUE"))
to_parallel <- TRUE
cat("Parallelization:", to_parallel, "\n")
# TODO: Add description here
to_debug <- FALSE
verbose_output <- if (to_debug) TRUE else FALSE


Parallelization: TRUE 


# Libraries


In [11]:
# Update the grouper
system("git submodule update --init --recursive")

# List, install (if applicable), and load packages
## Required packages
required_packages <- c(
  "data.table", "here", "tictoc", "stringr", "stringi", "lubridate",
  "profvis", "hash", "future", "future.apply", "knitr", "htmlwidgets",
  "parallelly", "stringdist", "parallel", "reticulate", "bigrquery",
  "jsonlite", "googleCloudStorageR", "haven", "fst", "httr", "ggplot2",
  "rmarkdown"
  # , "docstring", "progress" # Comma is here so if I uncomment this line it
  # automatically works without having to type or delete a comma after haven
)

# Additional packages to install via remotes (GitHub), if not available
github_packages <- c("r-lib/styler")

# Function to install and load packages quietly
install_and_load <- function(package) {
  if (!require(package, character.only = TRUE)) {
    message("Installing ", package)
    install.packages(package, dependencies = TRUE)
  } else {
    if (verbose_output) message("Loading ", package)
  }
  library(package, character.only = TRUE)
}

# Function to install packages from GitHub via remotes
install_from_github <- function(repo) {
  package_name <- basename(repo)
  if (!require(package_name, character.only = TRUE)) {
    if (!require("remotes", character.only = TRUE)) {
      install.packages("remotes")
    }
    message("Installing ", package_name, " from GitHub (", repo, ")")
    remotes::install_github(repo)
  } else {
    if (verbose_output) message("Loading ", package_name)
  }
  library(package_name, character.only = TRUE)
}

# Apply the function to each required package
message("Installing/loading required CRAN packages...")
invisible(
  suppressPackageStartupMessages(
    lapply(required_packages, install_and_load)
  )
)

# Install and load GitHub packages if not installed
message("Installing/loading required GitHub packages...")
invisible(
  suppressPackageStartupMessages(
    lapply(github_packages, install_from_github)
  )
)

# detect available threads
nthreads <- parallelly::availableCores()


Installing/loading required CRAN packages...

Installing/loading required GitHub packages...



# R Scripts


In [12]:
scripts_path <- here("data-cleaning/r_scripts_v2")

# List all R files in the directory with full paths, sorted by filename
r_files <- list.files(scripts_path, pattern = "\\.R$", full.names = TRUE)

# Source each file sequentially
for (file in r_files) {
  if (verbose_output) message(Sys.time(), " Sourcing: ", file)
  invisible(source(file))
}

message(year_to_load)


All directories exist.


Total Rows via cached object: 12757064

Utilizing 4 cores (8 threads)




Using virtual environment 'r-reticulate' ...


+ /home/resurreccion_cmc_gmail_com/.virtualenvs/r-reticulate/bin/python -m pip install --upgrade --no-user pip

2022



# Load Full Claims from GCS


In [13]:
# Load raw claims from GCS only if they don't exist on the VM yet
for (year in 2018:2023) {
  # Assign the correct file extension based on the year
  file_type <- if (year %in% c(2022:2023)) ".tsv" else ".csv"
  file_name <- paste0(full_claims_prefix, year, file_type)
  bq_name <- paste0(full_claims_bq_prefix, year, file_type)

  # Check if the file exists in the target directory
  file_path <- here(raw_claims_path, file_name)
  exists <- file.exists(file_path)

  # If the file does not exist, run the gsutil cp command
  if (!exists) {
    if (!is.null(gcp_proj) && gcp_proj == "drg-pipeline") {
      system(
        paste0(
          "cd .. && gsutil cp gs://phic-claims-raw/",
          bq_name, " ", raw_claims_path
        ),
        intern = FALSE, ignore.stderr = FALSE
      )
    } else {
      stop("Error: GCP Project is not null and is not drg-pipeline")
    }
  } else {
    message(paste(
      "File", file_name,
      "already exists in the target directory. Skipping download.\n"
    ))
  }
}


File claims_extract_CLAIMS 2018.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2019.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2020.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2021.csv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2022.tsv already exists in the target directory. Skipping download.


File claims_extract_CLAIMS 2023.tsv already exists in the target directory. Skipping download.




# Load Mapping Data


In [14]:
# Enable caching and printing options for data mapping
to_use_cache <- TRUE # Set to TRUE to enable saving and loading of .rds files
to_print_mapping_data <- TRUE # Set to TRUE to print mapping data tables

# Helper function to load data from cache or query from BigQuery if not cached
load_or_query <- function(query, var_name) {
  rds_path <- here(cache_path, "mapping", paste0(var_name, ".rds"))
  if (to_use_cache && file.exists(rds_path)) {
    if (verbose_output) message("Loading ", var_name, " from cache...")
    # Load data from .rds file if cache exists
    return(readRDS(rds_path))
  } else {
    if (verbose_output) message("Querying ", var_name, " from BigQuery...")
    # Query data from BigQuery if not cached
    # Query execution function (BigQuery to data.table)
    dt <- query_bq_to_dt(query)
    saveRDS(dt, rds_path) # Save queried data to .rds cache file
    return(dt)
  }
}

# Helper function to print all rows of a data.table if
# to_print_mapping_data is enabled
if (to_print_mapping_data) {
  print_all <- function(dt, title) {
    cat("\n---", title, "---\n") # Print table title
    print(dt, nrow = Inf) # Print all rows of the data.table
  }
}

# 1. Query and load the `grouper_v5.proc` table
# This table contains procedure codes and attributes
# like description, classification, and site
proc_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.proc`")
proc <- load_or_query(proc_query, "proc")
proc[, CODE := as.character(CODE)] # Ensure the CODE column is of character type

# 2. Query and load the `phic.acr_rvs_map` table
# This table maps RVS codes to ICD-9-CM codes,
# used for healthcare billing purposes
rvs_icd9_query <- paste0("SELECT * FROM `", gcp_proj, ".phic.acr_rvs_map`")
rvs_icd9 <- load_or_query(rvs_icd9_query, "rvs_icd9")

# Convert RVS and ICD9CM columns to character type
# and adjust ICD9CM for multiplication
rvs_icd9 <- rvs_icd9[, .(
  rvs = as.character(rvs),
  icd9cm = as.character(as.numeric(icd9cm) * 100)
)]

# Merge the RVS-ICD9 mapping with the proc table for DRG classification
rvs_icd9 <- merge(
  rvs_icd9,
  proc[, .(CODE, DRGUSE)], # Select CODE and DRGUSE columns for merging
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
  # Merge on icd9cm and CODE columns
)

# Filter and annotate DRG-related codes, removing unnecessary DRGUSE column
rvs_icd9 <- rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE][
  !is.na(rvs) & !is.na(icd9cm), -"DRGUSE"
]

# 3. Query and load `phic.acr_procedure` table
# This table contains RVS codes, relative value units (RVUs),
# and descriptions for procedures
acr_rvs_query <- paste0("SELECT * FROM `", gcp_proj, ".phic.acr_procedure`")
acr_rvs <- load_or_query(acr_rvs_query, "acr_rvs")

# 4. Query and load `grouper_v5.i10` table
# This table contains ICD-10 codes with DRG grouping data,
# including codes marked as "accepted" (ACCPDX = "Y")
i10_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.i10`")
tdrg_icd10 <- load_or_query(i10_query, "tdrg_icd10")
setkey(tdrg_icd10, "CODE") # Set the CODE column as key for efficient lookups

# Extract unique accepted ICD-10 codes for diagnosis
# (ACCPDX == "Y") and store in acc_pdx
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])

# Create an environment for quick lookup of accepted diagnosis codes
acc_pdx_env <- new.env(hash = TRUE, parent = emptyenv())
for (code in acc_pdx) {
  # Assign each accepted code to the environment
  assign(code, TRUE, envir = acc_pdx_env)
}

# 5. Query and load `icd.phl_icd10` table
# This table lists diseases and their corresponding
# ICD-10 codes specific to the Philippines
phl_icd10_query <- paste0("SELECT * FROM `", gcp_proj, ".icd.phl_icd10`")
phl_icd10 <- load_or_query(phl_icd10_query, "phl_icd10")

# Filter and process neoplasm codes by extracting
# specific codes from complex ICD-10 notations
neoplasms_dt_actual <- as.data.table(phl_icd10[
  # Select rows with '/' in icd10, indicating neoplasm codes
  grepl("/", icd10), .(icd10)
  # Extract relevant part
][, icd10 := sapply(strsplit(icd10, ","), function(x) trimws(x[2]))])


# 6. Query and load `grouper_v5.i10vx` table
# This table contains an expanded version of ICD-10 codes with validation flags
i10vx_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.i10vx`")
i10vx <- load_or_query(i10vx_query, "i10vx")
setkey(i10vx, "code") # Set the code column as key for efficient lookup
acc_icd <- unique(i10vx[, code]) # Extract unique ICD codes from this table
acc_icd_set <- unique(acc_icd)

# 7. Query and load `hci.temp_hci` table
# This table lists healthcare institutions with details
# like ownership, category, and location
hci_query <- paste0("SELECT * FROM `", gcp_proj, ".hci.temp_hci`")
hci <- load_or_query(hci_query, "hci")

# 8. Define global variables for use later in the script:
neoplasm_codes <- unique(neoplasms_dt_actual$icd10) # Unique neoplasm codes
covid_codes <- unique(covid_rvs) # Unique COVID-related codes
rvs_codes <- unique(acr_rvs$rvs) # Unique RVS codes

neoplasm_pattern <- paste0("(", paste(neoplasm_codes, collapse = "|"), ")")
covid_pattern <- paste0("(", paste(covid_codes, collapse = "|"), ")")
rvs_pattern <- paste0("(", paste(rvs_codes, collapse = "|"), ")")

phil_icds <- unique(gsub("[^A-Za-z0-9]", "", phl_icd10[!grepl("/", icd10), icd10]))
icd_codes <- unique(tdrg_icd10$CODE)

# Function to create an environment from a vector of unique values
create_env_from_vector <- function(vec) {
  env <- new.env(parent = emptyenv())
  list2env(setNames(as.list(rep(TRUE, length(vec))), vec), envir = env)
  return(env)
}

# 1. Create environment for `proc` table data if specific values are needed
# Here we assume `proc$CODE` is the field of interest
proc_env <- create_env_from_vector(proc$CODE)

# 2. Create environment for `rvs_icd9` table data based on `rvs` and `icd9cm`
rvs_env <- create_env_from_vector(rvs_icd9$rvs)
icd9cm_env <- create_env_from_vector(rvs_icd9$icd9cm)

# 3. Environment for `acr_rvs` table (assuming `rvs` is the field of interest)
acr_rvs_env <- create_env_from_vector(acr_rvs$rvs)

# 4. Environment for accepted ICD-10 codes (from `tdrg_icd10`)
acc_pdx_env <- create_env_from_vector(acc_pdx)

# 5. Environment for `phl_icd10` ICD-10 codes (e.g., neoplasm codes)
phl_icd10_env <- create_env_from_vector(phl_icd10$icd10)

# 6. Environment for expanded ICD-10 codes (`i10vx`)
acc_icd_env <- create_env_from_vector(i10vx$code)

# 7. Environment for `hci` table data if needed for specific fields (e.g., `id_hci`)
# Assuming `hci$id_hci` is the identifier of interest
hci_env <- create_env_from_vector(hci$id_hci)

# 8. Other specific environments for global variables
neoplasm_env <- create_env_from_vector(neoplasm_codes)
covid_env <- create_env_from_vector(covid_codes)
rvs_codes_env <- create_env_from_vector(rvs_codes)
phil_icds_env <- create_env_from_vector(phil_icds)
icd_codes_env <- create_env_from_vector(icd_codes)

# Combine COVID and neoplasm codes into a single environment
covid_neoplasm_codes <- unique(c(covid_codes, neoplasm_codes))
covid_neoplasm_env <- create_env_from_vector(covid_neoplasm_codes)

# Combine COVID, RVS, and neoplasm codes into a single environment for efficient lookup
covid_rvs_neoplasm_codes <- unique(c(covid_codes, rvs_codes, neoplasm_codes))
covid_rvs_neoplasm_env <- create_env_from_vector(covid_rvs_neoplasm_codes)


# Create a combined regular expression pattern to match COVID,
# RVS, and neoplasm codes in data processing
covid_rvs_neoplasm_pattern <- paste(
  c(covid_codes, rvs_codes, neoplasm_codes),
  collapse = "|"
)
# if (to_print_mapping_data) print(covid_rvs_neoplasm_pattern)
# # Print regex pattern if enabled

# Helper function to save all specified data tables into a
# single text file for debugging
save_all_data_to_file <- function(file_path, ...) {
  args <- list(...)
  sink(file_path) # Redirect output to the specified file
  cat("\n--- All Data Tables in One View ---\n") # Header for the file
  for (name in names(args)) {
    cat("\n---", name, "---\n") # Print table name as a header within the file
    # Print all rows of each data.table
    print(args[[name]], nrow = Inf, max.print = Inf)
  }
  sink() # Stop redirecting output to the file
  if (verbose_output) message("All data tables saved to ", file_path) # Confirmation message
}

# Set the file path for the output text file, where all
# data tables will be saved
output_file <- here(debug_path, "mapping_data.txt")

# If enabled, save all processed data tables to a single specified
# file for debugging and verification.
# Each table represents a different aspect of the medical coding,
# classification, and healthcare provider data.
if (to_print_mapping_data) {
  options(max.print = 999999)
  save_all_data_to_file(
    output_file,

    # Data table containing procedure codes and attributes
    # for each procedure code.
    # Columns include CODE (unique procedure identifier),
    # DRGUSE (flag indicating if the procedure is used for DRG grouping),
    # and several other attributes related to procedure
    # classification, gender applicability, site, and level of care.
    grouper_v5_proc = proc,

    # Mapping between ICD-9-CM codes and RVS (Relative Value Scale)
    # codes used in medical billing.
    # Includes `is_drg` column to mark codes used in DRG grouping
    # after merging with the `grouper_v5.proc` table.
    # This table links standard ICD-9 procedure codes to specific
    # RVS codes for billing purposes.
    phic_acr_rvs_map = rvs_icd9,

    # Table of RVS codes and their associated RVU (Relative Value Units)
    # which represent the value of a procedure.
    # Also includes a detailed description of each procedure, such as type,
    #  category, or specific details about the procedure.
    # This table is essential for understanding the cost/value of each
    # RVS-coded procedure in medical billing.
    phic_acr_procedure = acr_rvs,

    # ICD-10 table with additional classification details relevant for
    #  DRG (Diagnosis Related Group) mapping.
    # Columns include CODE (ICD-10 diagnosis code), ACCPDX
    # (accepted primary diagnosis flag), and other grouping
    # variables like MDC (Major Diagnostic Category) and CC
    # (Complication/Comorbidity), which help classify the severity or
    # complexity of cases for healthcare reimbursement.
    grouper_v5_i10 = tdrg_icd10,

    # A unique list of ICD-10 codes flagged as ACCPDX (accepted
    # primary diagnosis codes) for use in DRG classification.
    # This list is derived from `grouper_v5.i10` and is used as
    # a quick reference to check if a diagnosis is eligible as a primary code.
    acc_pdx = acc_pdx,

    # Data specific to the Philippines for ICD-10 codes, containing
    # disease names and the corresponding ICD-10 codes.
    # This table includes the field `remarks`, which provides
    # special notes or guidance for each code, such as diagnostic
    # instructions or clarifications. This dataset is used to
    # manage and classify diseases in line with local health regulations.
    icd_phl_icd10 = phl_icd10,

    # Processed subset of neoplasm codes extracted from
    # `icd.phl_icd10`.
    # Contains ICD-10 codes specifically formatted to represent
    # malignant, benign, and other tumor types.
    # Useful for oncology-specific mappings in DRG processing
    # or cancer-related case management.
    neoplasms_dt_actual = neoplasms_dt_actual,

    # Expanded ICD-10 dataset with validation flags indicating
    # whether each code is valid.
    # Includes columns such as `validcode` (flag for validation status)
    # and `todel` (marker for codes that may need removal).
    # This dataset helps verify the validity of ICD-10 codes and
    #  manage code deprecation or updates.
    grouper_v5_i10vx = i10vx,

    # List of unique ICD-10 codes extracted from `grouper_v5.i10vx`
    # for quick access.
    # Acts as a condensed reference of all validated ICD-10 codes
    # available in the `grouper_v5.i10vx` dataset.
    # Useful for ensuring consistency and accuracy in ICD-10 code
    # usage across processes.
    acc_icd = acc_icd,

    # A directory of healthcare institutions (HCI), containing
    # detailed information about each provider,
    # including their institution name, ownership type (e.g.,
    # government, private), category, geographical details,
    # and provider classification. This table enables linkage
    # between clinical data and provider-specific data,
    # allowing for enhanced reporting and analytics on healthcare
    # service providers.
    hci_temp_hci = hci
  )
  # print(rvs_pattern)
  options(max.print = 1000)
}


# Read Data

In [15]:
# Read the RDS files for stata and thai
stata <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata", ".rds")))
thai <- readRDS(here(checkpoint_6_path, paste0(checkpoint_6_prefix, year_to_load, suffix, ".rds")))

# # Remove duplicate rows based on id_series in both datasets
# stata <- unique(stata, by = "id_series")
# thai <- unique(thai, by = "id_series")

# Ensure both data.tables have the same key for joining
setkey(stata, id_series)
setkey(thai, id_series)

# Perform a full join (merge all rows from both data.tables)
joined_data <- merge(stata, thai, by = "id_series", all = TRUE)

# Check the structure of the merged data
str(joined_data)

# print(head(joined_data, 100))

saveRDS(joined_data, here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata_final", ".rds")))


Classes ‘data.table’ and 'data.frame':	4613644 obs. of  49 variables:
 $ id_series        : chr  "0000010000101112210000" "0000010000101302210000" "0000010000101402210000" "0000010000101502210000" ...
 $ id_year          : num  2022 2022 2022 2022 2022 ...
 $ id_pin           : chr  "9AE47CE6B52854700308F5017B2FCCD1" "B5A124BA1DA4EF208D1035B27BD7EE5E" "A576DEECB51CAD9D14EC07A2B63B2EE0" "D5150B0AB29691421ED721D2E22DD81B" ...
 $ id_hci           : chr  "520101" "520101" "490829" "460101" ...
 $ id_hcp           :List of 4613644
  ..$ : chr "67150"
  ..$ : chr "3286"
  ..$ : chr "54720"
  ..$ : chr "34812"
  ..$ : chr  "25599" "26632" "6383"
  ..$ : chr "27186"
  ..$ : chr "4917"
  ..$ : chr "70190"
  ..$ : chr "24515"
  ..$ : chr "3982"
  ..$ : chr "67150"
  ..$ : chr "35383"
  ..$ : chr "1201"
  ..$ : chr "24515"
  ..$ : chr "67150"
  ..$ : chr "54720"
  ..$ : chr "27411"
  ..$ : chr "67150"
  ..$ : chr "24515"
  ..$ : chr "38029"
  ..$ : chr "69194"
  ..$ : chr "24515"
  ..$ : chr "876

# .dta Export and Upload


In [16]:
result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata_final", ".rds")))

# Define a function to convert list columns to character columns
convert_list_to_character <- function(dt) {
  # Loop through all columns in the data.table
  for (col in names(dt)) {
    if (is.list(dt[[col]])) {
      dt[, (col) := sapply(dt[[col]], function(x) {
        if (is.null(x) || all(is.na(x))) {
          return(NA_character_) # Return NA if the list is empty or all values are NA
        } else {
          return(paste(sort(x), collapse = ",")) # Sort and concatenate
        }
      })]
    }
  }
}

# Apply the conversion function to your data.table 'result'
convert_list_to_character(result)

# Export to Stata (.dta) format
write_dta(as.data.frame(result), here(checkpoint_10_path, paste0(checkpoint_10_prefix, year_to_load, suffix, ".dta")))

gcs_auth(email = gcs_email)
gcs_upload(
  file = here(checkpoint_10_path, paste0(checkpoint_10_prefix, year_to_load, suffix, ".dta")),
  bucket = gcs_bucket,
  name = paste0(gcs_spc_fpath, "/", paste0(checkpoint_10_prefix, year_to_load, suffix, ".dta")),
  predefinedAcl = "bucketLevel"
)

message(".dta has now been uploaded")


ℹ 2024-11-11 08:01:49.77714 > File size detected as  3.5 Gb

ℹ 2024-11-11 08:01:49.843928 > Found resumeable upload URL:  https://www.googleapis.com/upload/storage/v1/b/phic-claims-checkpoints/o/?uploadType=resumable&name=spc%2Fstata2022_full_.dta&upload_id=AHmUCY2Oqz3HXEBABefxmDCTMIyodTg0TSZSqZoMCTySCrnwJFad5JrduXa7crO8X5eDGVM4wRd_NdC14BN0KxzZBRzGDlEzfo5QZW9ZknITf1i6gQ

.dta has now been uploaded



# BQ Upload


In [17]:
# result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata_final", ".rds")))


In [18]:
# bq_table <- paste0("spc_", year_to_load)

# # Check if the table should be dropped and replaced
# tryCatch(
#   {
#     bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table))
#     message("Table dropped successfully.\n")
#   },
#   error = function(e) {
#     # If the table does not exist, just continue
#     if (grepl("Not found", e, ignore.case = TRUE)) {
#       message("Table does not exist, nothing to drop.\n")
#     } else {
#       # If it's a different error, re-throw the error
#       stop(e)
#     }
#   }
# )

# # Attempt to create the table
# tryCatch(
#   {
#     bq_table_create(
#       bq_table(gcp_proj, bq_dataset, bq_table),
#       fields = fromJSON(here(
#         "data-cleaning/r_scripts_v2",
#         "bq_schema_spc.json"
#       ), simplifyDataFrame = FALSE)
#     )
#     message("Table created successfully.\n")
#   },
#   error = function(e) {
#     # Check if the error message indicates that the table already exists
#     if (grepl("already exists", e, ignore.case = TRUE)) {
#       message("Table already exists. Skipping creation and upload.")
#     } else {
#       # If it's a different error, re-throw the error
#       stop(e)
#     }
#   }
# )

# # Upload to BQ only if table is empty
# if (to_write) {
#   chunk_size <- 250000 # Adjust the chunk size based on memory availability
#   num_chunks <- ceiling(nrow(result) / chunk_size)

#   for (i in seq_len(num_chunks)) {
#     cat(paste("\rUploading chunk no.:", i))
#     flush.console()
#     chunk <- result[
#       ((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(result)),
#     ]

#     bq_table_upload(
#       bq_table(gcp_proj, bq_dataset, bq_table),
#       values = chunk,
#       write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
#     )
#     cat(paste("\rFinished uploading chunk no.:", i))
#     flush.console()
#   }
# }
